# loudkit — hear a voice

Text to speech that runs on your own machine, in ten languages, with voice
cloning from ten seconds of audio.

This notebook downloads the model, speaks a sentence, and clones a voice.
Runtime → Run all, or step through it. A GPU runtime is faster but not
required — the CPU path is the same algorithm and the same audio.

[Repository](https://github.com/loudreader/loudkit) ·
[Voices](https://github.com/loudreader/loudkit/blob/main/VOICES.md) ·
[Model card](https://github.com/loudreader/loudkit/blob/main/docs/MODEL_CARD.md)

In [ ]:
# The `hub` extra is what lets `load()` take a model name instead of a path.
%pip install -q "loudkit[torch,audio,hub]"

## Speak

The first call downloads about 750 MB and caches it, so a second notebook on
the same machine starts instantly. Load the engine once and keep it. Cloning
later adds the separate 523 MB enrollment checkpoint.

In [ ]:
from IPython.display import Audio

import loudkit as lk

MODEL = "loudreader/loudr-1"

engine = lk.load(MODEL)
narrator = lk.voice("joe", repo=MODEL)

result = engine.synthesize("Hello from a machine you control.", narrator, seed=7)
print(result)
Audio(result.audio, rate=result.sample_rate)

## The same seed gives the same audio

Not a slogan — the engine refuses to start if two of its components disagree
about what to compute, and five independent implementations (Python, Swift,
Go, Rust, TypeScript) are held to one conformance fixture.

Change the seed and you get a different, equally valid reading.

In [ ]:
import numpy as np

again = engine.synthesize("Hello from a machine you control.", narrator, seed=7)
print("bit-identical:", np.array_equal(result.audio, again.audio))

different = engine.synthesize("Hello from a machine you control.", narrator, seed=8)
print(
    "a different seed is a different reading:",
    not np.array_equal(result.audio, different.audio),
)
Audio(different.audio, rate=different.sample_rate)

## A whole passage

One window is about ten seconds. `synthesize_long` splits at sentence
boundaries and carries prosody across the joins, so a paragraph does not
restart its pitch contour at every break. `stream` is the same synthesis
delivered chunk by chunk, if you want to start playing before it finishes.

In [ ]:
passage = (
    "The first sentence sets the scene and runs on for a while. "
    "The second follows it and is no shorter than the first one was. "
    "The third exists so that the splitter has somewhere to breathe."
)
long = engine.synthesize_long(passage, narrator, seed=7)
print(f"{long.duration:.1f}s from {len(long.tokens)} tokens")
Audio(long.audio, rate=long.sample_rate)

## Other languages

Voices ship for nine. **Quality has been evaluated for English only** — the
rest will be read, not necessarily spoken well, so listen before you ship one.

In [ ]:
gosia = lk.voice("gosia", repo=MODEL)
polish = engine.synthesize_long(
    "Pobierz aplikację i posłuchaj, jak brzmi ten głos po polsku.",
    gosia,
    seed=7,
    language="pl",
)
Audio(polish.audio, rate=polish.sample_rate)

## Clone a voice

Ten seconds of clean audio. The result is a ~150 KB file of tensors, not a
model, so you can copy it around like any other file.

**Consent is yours to obtain.** Cloning someone's voice without it is not a
technical question — see
[RESPONSIBLE_USE.md](https://github.com/loudreader/loudkit/blob/main/RESPONSIBLE_USE.md).

In [ ]:
%pip install -q "loudkit[enroll]"

# Record in the browser, or upload a file with the Files pane on the left and
# point `SOURCE` at it. Anything clean and mono works; ~10 s is plenty.
SOURCE = None  # e.g. "/content/my-recording.wav"

if SOURCE:
    mine = lk.enroll(SOURCE, MODEL, name="my-voice")
    mine.save("my-voice.safetensors")

    spoken = engine.synthesize_long("This is my own voice, running locally.", mine, seed=7)
    display(Audio(spoken.audio, rate=spoken.sample_rate))
else:
    print("Set SOURCE to a recording to try cloning.")

## Where to go next

- **Another language than Python** — Swift, Go, Rust and TypeScript are full
  ports, not wrappers, held to the same conformance fixture. See
  [`docs/guides/`](https://github.com/loudreader/loudkit/tree/main/docs/guides).
- **A local server** — `loudkit serve` keeps the model warm and answers HTTP;
  `loudkit mcp` does the same for agents.
- **What it promises** — [the measured parity
  table](https://github.com/loudreader/loudkit/blob/main/docs/parity-measured.md).